In [ ]:
import os, sys, subprocess, shutil
print("SHOT LOOKDEV: environment check")
print("Free working space GB:", round(shutil.disk_usage("/kaggle/working").free / 1e9, 2))
subprocess.run(["nvidia-smi"], check=False)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "diffusers>=0.33.0", "transformers>=4.48", "accelerate>=0.34", "safetensors", "sentencepiece", "ftfy", "pillow"])
print("Lookdev dependencies ready.")


In [ ]:
import hashlib, json, time
from pathlib import Path
import torch
from PIL import Image, ImageDraw, ImageFont
from diffusers import AutoPipelineForText2Image

PLAN = json.loads("{\"title\": \"Machine Learning vs Deep Learning\", \"target_duration_seconds\": 120, \"model\": \"Wan-AI/Wan2.1-T2V-1.3B-Diffusers\", \"generation_mode\": \"text-to-video\", \"shot_duration_seconds\": 5, \"shot_count\": 24, \"output_fps\": 16, \"generation_resolution\": \"832x480\", \"edit_notes\": \"Every picture is a newly generated moving video shot. Do not use the storyboard panel images as footage. Keep all factual labels and diagrams as deterministic post-production overlays; never ask the video model to render technical text.\", \"negative_prompt\": \"still image, frozen motion, slideshow, static camera, cartoon, illustration, painting, text, letters, labels, subtitles, watermark, logos, flicker, abrupt cuts, blurry, low quality, extra limbs, malformed hands, distorted faces, duplicate subjects, impossible anatomy\", \"shots\": [{\"id\": \"S01\", \"start\": 0, \"beat\": 1, \"visual\": \"Opening: host and cat\", \"prompt\": \"Photorealistic educational documentary footage in a welcoming home studio. A friendly young science presenter in a navy hoodie turns toward camera and gestures hello while a ginger cat steps onto the desk. Natural blink and hand movement, soft daylight, slow camera dolly in, one continuous shot, no text or logos.\"}, {\"id\": \"S02\", \"start\": 5, \"beat\": 1, \"visual\": \"Examples power learning\", \"prompt\": \"Photorealistic close shot of a ginger cat and a golden retriever meeting in a sunlit room. The cat looks up, flicks its ears, and the dog gently tilts its head and sniffs. Fur moves naturally, camera makes a smooth low tracking move, one continuous live-action-style shot, no text.\"}, {\"id\": \"S03\", \"start\": 10, \"beat\": 2, \"visual\": \"AI branches into methods\", \"prompt\": \"A cinematic macro shot of a glowing blue neural data stream entering a transparent glass control room, then physically branching into two separate flowing streams, one blue and one violet. Tiny light particles travel continuously through the branching paths as the camera orbits slowly, crisp educational science visualization, no letters or text.\"}, {\"id\": \"S04\", \"start\": 15, \"beat\": 2, \"visual\": \"Learning from examples\", \"prompt\": \"Photorealistic hands arrange many printed cat and dog photographs across a large table while a compact computer vision camera scans each image. The photos slide into neat stacks and colored light pulses travel toward a screen, overhead camera glides sideways, clear real-world educational documentary style, no readable text.\"}, {\"id\": \"S05\", \"start\": 20, \"beat\": 3, \"visual\": \"Cat example\", \"prompt\": \"Photorealistic orange tabby cat walking across a warm living-room window ledge, pausing to look toward the camera, whiskers twitching and tail moving. Slow close-up tracking shot from the cat's eye to its face, natural fur and soft daylight, one continuous shot, no text.\"}, {\"id\": \"S06\", \"start\": 25, \"beat\": 3, \"visual\": \"Dog example\", \"prompt\": \"Photorealistic golden retriever trots through a leafy park toward camera, slows, sniffs a flower, and turns its head. Natural paw movement and fur motion, low tracking camera follows smoothly, sunlit background with shallow depth of field, one continuous shot, no text.\"}, {\"id\": \"S07\", \"start\": 30, \"beat\": 4, \"visual\": \"Human-selected features\", \"prompt\": \"Photorealistic close-up of a researcher examining a calm golden retriever, gently pointing to its ears, coat, body outline, and paws while making notes on a transparent tablet. The dog blinks and turns its head; camera makes a small smooth arc, warm laboratory lighting, no readable writing.\"}, {\"id\": \"S08\", \"start\": 35, \"beat\": 4, \"visual\": \"Features become a decision\", \"prompt\": \"A photorealistic close view of a computer vision workstation analyzing a moving dog video. Clean colored tracking points lock onto the ears, fur, and body outline as the footage advances, then a simple blue decision path illuminates. Camera slowly pushes toward the monitor, screen contains abstract shapes only, no text.\"}, {\"id\": \"S09\", \"start\": 40, \"beat\": 5, \"visual\": \"Deep neural network\", \"prompt\": \"A cinematic 3D visualization of a raw cat photograph breaking into luminous pixels that flow through several layered neural-network grids, each layer lighting up in sequence. Data pulses move continuously from left to right as the camera flies between layers, saturated blue and magenta lighting, no words or labels.\"}, {\"id\": \"S10\", \"start\": 45, \"beat\": 5, \"visual\": \"Automatic feature learning\", \"prompt\": \"Photorealistic macro animation of a cat image being analyzed by a deep-learning system: raw pixels become edge maps, then eyes, whiskers, and a recognizable cat silhouette. Each transformation flows smoothly across transparent glass panels while the camera tracks alongside, vivid and technically grounded, no text.\"}, {\"id\": \"S11\", \"start\": 50, \"beat\": 6, \"visual\": \"Early layers detect edges\", \"prompt\": \"Extreme macro science visualization inside an artificial neural network. Small blue nodes activate in a wave, tracing edges and simple shapes from a moving image, then passing signals forward. The camera travels through the first glowing layers, visible flowing particles, realistic volumetric lighting, no text.\"}, {\"id\": \"S12\", \"start\": 55, \"beat\": 6, \"visual\": \"Later layers recognize objects\", \"prompt\": \"The camera continues through deeper luminous network layers as simple edges combine into eyes, whiskers, a cat face, then a complete recognizable cat. Bright nodes pulse in order and data streams cascade between them, cinematic volumetric blue and violet light, no written labels.\"}, {\"id\": \"S13\", \"start\": 60, \"beat\": 7, \"visual\": \"Medical imaging\", \"prompt\": \"Photorealistic hospital radiology room. A radiologist studies a chest X-ray on a large monitor while a 3D scan rotates slowly and subtle analysis highlights move over the image. The clinician adjusts the view and leans closer, calm professional lighting, camera pushes in, no patient identifiers or readable text.\"}, {\"id\": \"S14\", \"start\": 65, \"beat\": 7, \"visual\": \"Face recognition\", \"prompt\": \"Photorealistic public-building entrance with people walking through naturally. A computer-vision camera follows one volunteer subject and clean green tracking corners move with the face while the person turns and smiles. Respectful neutral depiction, smooth lateral tracking shot, no readable signs or identity claims.\"}, {\"id\": \"S15\", \"start\": 70, \"beat\": 7, \"visual\": \"Self-driving car\", \"prompt\": \"Photorealistic point-of-view from a self-driving car moving through a city street at golden hour. Cars, cyclists, and pedestrians move naturally; subtle colored detection outlines follow objects as the vehicle slows and turns safely. Smooth forward camera motion, realistic road reflections, no text.\"}, {\"id\": \"S16\", \"start\": 75, \"beat\": 8, \"visual\": \"Choosing the right method\", \"prompt\": \"Photorealistic engineer at a glass wall compares a small neat spreadsheet of simple data with a large stream of images, sound waves, and video clips. The simple chart stays compact while complex data flows toward a glowing neural network; camera sweeps smoothly across the wall, no readable writing.\"}, {\"id\": \"S17\", \"start\": 80, \"beat\": 8, \"visual\": \"Small data favors simple baselines\", \"prompt\": \"A close-up of a researcher selecting a few clear measurements on a tablet, then feeding them into a compact transparent machine-learning pipeline. Three feature sliders move and a simple result indicator changes color; hands remain natural, gentle camera slide, clean documentary look, no words.\"}, {\"id\": \"S18\", \"start\": 85, \"beat\": 8, \"visual\": \"Complex data needs richer models\", \"prompt\": \"A dynamic wall of images, audio waveforms, and short video fragments streams into a large layered neural network. Thousands of colorful particles cascade through the layers and converge into one steady output beam while the camera circles the structure, cinematic high-detail educational visualization, no text.\"}, {\"id\": \"S19\", \"start\": 90, \"beat\": 9, \"visual\": \"Key takeaway\", \"prompt\": \"Photorealistic close shot of a hand writing two simple diagrams on a clean transparent board: a compact feature-to-result flow and a larger layered neural network. The lines draw themselves smoothly and small data points animate through them; slow camera move, leave clear empty spaces for post-production labels, no readable handwriting.\"}, {\"id\": \"S20\", \"start\": 95, \"beat\": 9, \"visual\": \"Compare on unseen data\", \"prompt\": \"Two animated model pipelines receive a fresh set of images they have not seen before. One compact pipeline and one layered neural network each produce a result while a scientist compares their changing accuracy and cost indicators on a glass display. Fluid camera orbit, indicators are abstract shapes only, no text.\"}, {\"id\": \"S21\", \"start\": 100, \"beat\": 10, \"visual\": \"Both methods solve problems\", \"prompt\": \"Cinematic transition from a compact machine-learning model and a deep neural network toward real-world scenes: a doctor reviewing a scan, a speech waveform becoming a conversation, and a vehicle navigating safely. Smooth connected camera move, warm and cool color contrast, no text.\"}, {\"id\": \"S22\", \"start\": 105, \"beat\": 10, \"visual\": \"Learning from data\", \"prompt\": \"A photorealistic science visualization of two different paths through a glowing data landscape, one compact and one deeply layered, both reaching the same clear solution at the horizon. Data particles keep moving throughout; camera flies forward between the paths, vivid blue and amber lighting, no labels.\"}, {\"id\": \"S23\", \"start\": 110, \"beat\": 11, \"visual\": \"Final ML and DL comparison\", \"prompt\": \"A luminous transparent brain made from moving blue and violet neural fibers floats in a dark cinematic studio. Two streams labeled only by distinct colors flow into the same network and converge into a bright result; the camera makes a slow half-orbit, dense light particles move continuously, no text.\"}, {\"id\": \"S24\", \"start\": 115, \"beat\": 12, \"visual\": \"Friendly close\", \"prompt\": \"Photorealistic young science presenter in a navy hoodie smiles and waves goodbye to the camera in the same warm educational studio, a ginger cat beside them stretches and jumps down from the desk. Camera gently pulls back, natural movement, bright inviting finish, leave open space on the right for post-production call-to-action graphics, no text or logos.\"}]}")
MODEL_ID = "stabilityai/sdxl-turbo"
OUT = Path("/kaggle/working/lookdev")
OUT.mkdir(parents=True, exist_ok=True)
print("MODEL", MODEL_ID, "SHOTS", len(PLAN["shots"]))
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, variant="fp16", use_safetensors=True
)
pipe.enable_model_cpu_offload(gpu_id=0)
font = ImageFont.load_default()
tiles = []
manifest = {"model": MODEL_ID, "resolution": "672x384", "steps": 2, "shots": []}
for index, shot in enumerate(PLAN["shots"]):
    prompt = shot["prompt"] + " vivid natural color, cinematic documentary lighting, clear readable composition"
    seed = 51400 + index
    generator = torch.Generator(device="cuda").manual_seed(seed)
    started = time.time()
    with torch.inference_mode():
        image = pipe(prompt=prompt, height=384, width=672, num_inference_steps=2,
                     guidance_scale=0.0, generator=generator).images[0]
    path = OUT / f"{shot['id']}.png"
    image.save(path)
    hsv = image.convert("HSV")
    saturation = sum(hsv.getchannel("S").getdata()) / (255 * image.width * image.height)
    manifest["shots"].append({"id": shot["id"], "visual": shot["visual"], "file": path.name,
                              "seed": seed, "mean_saturation": round(saturation, 4),
                              "elapsed_seconds": round(time.time() - started, 2),
                              "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
    tile = image.resize((336, 192))
    draw = ImageDraw.Draw(tile)
    draw.rectangle((0, 0, 120, 20), fill=(0, 0, 0))
    draw.text((5, 4), f"{shot['id']} | {shot['visual'][:31]}", font=font, fill=(255, 255, 255))
    tiles.append(tile)
    print("LOOKDEV_DONE", shot["id"], "saturation", round(saturation, 3),
          "elapsed", round(time.time() - started, 1))

cols, rows = 4, 6
sheet = Image.new("RGB", (cols * 336, rows * 192), (12, 16, 24))
for i, tile in enumerate(tiles):
    sheet.paste(tile, ((i % cols) * 336, (i // cols) * 192))
sheet.save("/kaggle/working/lookdev_contact_sheet.jpg", quality=92)
Path("/kaggle/working/lookdev_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("LOOKDEV_COMPLETE", len(tiles), "contact sheet ready")
